# Reproduce Results

This notebook **loads pre-trained models from disk** and evaluates them on the
train / validation / test splits.  No training happens here.

Metrics reported:
- **ROC-AUC** (`sklearn.metrics.roc_auc_score`)
- **Precision** (`sklearn.metrics.precision_score`)
- **Recall** (`sklearn.metrics.recall_score`)
- **F1** (`sklearn.metrics.f1_score`)

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import utils
from utils import (
    load_object,
    get_combined_features,
    evaluate_model,
)

## 1. Data loading and splitting

The exact same random seed and split ratios as `train_models.ipynb` are used
so that train / val / test sets are identical across notebooks.

In [2]:
DATA_PATH = os.path.expanduser("~/Datasets/QuoraQuestionPairs/quora_data.csv")
quora_df = pd.read_csv(DATA_PATH)

A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df,  test_size=0.05, random_state=123)

print(f'train_df.shape = {train_df.shape}')
print(f'val_df.shape   = {val_df.shape}')
print(f'test_df.shape  = {test_df.shape}')

y_train = train_df["is_duplicate"].values
y_val   = val_df["is_duplicate"].values
y_test  = test_df["is_duplicate"].values

train_df.shape = (291897, 6)
val_df.shape   = (15363, 6)
test_df.shape  = (16172, 6)


## 2. Load models and vectorizers

In [3]:
MODELS_DIR = "models"

count_vectorizer    = load_object(os.path.join(MODELS_DIR, "count_vectorizer.pkl"))
tfidf_vectorizer    = load_object(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"))
baseline_logistic   = load_object(os.path.join(MODELS_DIR, "baseline_logistic.pkl"))
sbert_logistic   = load_object(os.path.join(MODELS_DIR, "sbert_logistic.pkl"))
sbert_graph_logistic = load_object(os.path.join(MODELS_DIR, "sbert_graph_logistic.pkl"))

print("All models and vectorizers loaded successfully.")

All models and vectorizers loaded successfully.


## 3. Feature extraction

- **Baseline**: sparse BoW matrix (CountVectorizer, unigrams)
- **Improved**: BoW + 5 handcrafted similarity features (see `utils.py`)

In [4]:
print("\nExtracting combined (BoW + handcrafted) features...")
X_train_comb = get_combined_features(train_df, count_vectorizer, tfidf_vectorizer)
X_val_comb   = get_combined_features(val_df,   count_vectorizer, tfidf_vectorizer)
X_test_comb  = get_combined_features(test_df,  count_vectorizer, tfidf_vectorizer)
print("Baseline model loaded")

X_train_sbert = np.load(os.path.join(MODELS_DIR, "sbert_X_train.npy"))  
X_val_sbert   = np.load(os.path.join(MODELS_DIR, "sbert_X_val.npy"))    
X_test_sbert  = np.load(os.path.join(MODELS_DIR, "sbert_X_test.npy"))
print("SBERT model and feature matrices loaded.") 

X_train_sg = np.load(os.path.join(MODELS_DIR, "sbert_graph_X_train.npy")) 
X_val_sg   = np.load(os.path.join(MODELS_DIR, "sbert_graph_X_val.npy"))    
X_test_sg  = np.load(os.path.join(MODELS_DIR, "sbert_graph_X_test.npy")) 
print("SBERT+Graph model and feature matrices loaded.")


y_train = train_df["is_duplicate"].values
y_val   = val_df["is_duplicate"].values


Extracting combined (BoW + handcrafted) features...
Baseline model loaded
SBERT model and feature matrices loaded.
SBERT+Graph model and feature matrices loaded.


## 4. Evaluation — ROC-AUC, Precision, Recall, F1

In [6]:
rows = []

# Improved model on all three splits
rows.append(evaluate_model(baseline_logistic, X_train_comb, y_train, "baseline", "train"))
rows.append(evaluate_model(baseline_logistic, X_val_comb,   y_val,   "baseline", "val"))
rows.append(evaluate_model(baseline_logistic, X_test_comb,  y_test,  "baseline", "test"))

rows.append(evaluate_model(sbert_logistic, X_train_sbert, y_train, "sbert", "train"))
rows.append(evaluate_model(sbert_logistic, X_val_sbert,   y_val,   "sbert", "val"))
rows.append(evaluate_model(sbert_logistic, X_test_sbert,  y_test,  "sbert", "test"))

rows.append(evaluate_model(sbert_graph_logistic, X_train_sg, y_train, "sbert_graph", "train"))
rows.append(evaluate_model(sbert_graph_logistic, X_val_sg,   y_val,   "sbert_graph", "val"))
rows.append(evaluate_model(sbert_graph_logistic, X_test_sg,  y_test,  "sbert_graph", "test"))

results_df = pd.DataFrame(rows)
results_df = results_df.set_index(["model", "split"])
print("Final results")
display(results_df)

Final results


roc_auc  precision  recall      f1  accuracy
model       split                                              
baseline    train   0.9419     0.8167  0.8324  0.8245    0.8694
            val     0.9007     0.7542  0.7662  0.7602    0.8216
            test    0.9061     0.7624  0.7638  0.7631    0.8238
sbert       train   0.8988     0.7567  0.7430  0.7498    0.8172
            val     0.8939     0.7574  0.7277  0.7423    0.8136
            test    0.8983     0.7592  0.7353  0.7471    0.8151
sbert_graph train   0.9451     0.8457  0.7866  0.8151    0.8684
            val     0.9448     0.8522  0.7752  0.8119    0.8675
            test    0.9443     0.8499  0.7729  0.8096    0.8650